# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*' 'pyyaml==6.0.*'

    # mount colab folder
    #from google.colab import drive
    #drive.mount('/content/drive')
    #base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

    # directly download the prepared dataset and shared helpers
    # (Colab only ever gets the 'small' set -- 'full' is local-only, see 1-preparation.ipynb)
    ! mkdir -p 'results'
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-image.zip
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-labels.csv
    ! wget -nv https://raw.githubusercontent.com/mgmalheiros/vision/master/process/counting/common.py
    base_folder = pathlib.Path('./')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL,pandas,scipy,yaml'))

# Loading

In [ ]:
# load the prepared dataset
dataset_size = 'full-'  # 'tiny-' (10) or ''(100) or 'full-'(6000+)

import common

images = common.load_prepared_images(base_folder / 'prepared' / f'1-coins-{dataset_size}image.zip')
df = common.load_labels(base_folder / 'prepared' / f'1-coins-{dataset_size}labels.csv')

print(f'{len(images)} images loaded')
df.describe()

# Region-based Method (watershed)
Build two marker classes from raw intensity, flood the Sobel elevation map from those
markers with `watershed`, then clean up and count the region that corresponds to coins
(darker than the background here — see the note in the function below).

> **Note on `low_marker`/`high_marker`:** `markers[image > low_marker] = 1` runs before
> `markers[image < high_marker] = 2`, so any pixel in `(low_marker, high_marker)` is
> first marked `1` and then immediately overwritten to `2`. With the defaults (50, 75)
> that collapses the intended three-zone map (background / unmarked / foreground) into
> a plain split at `high_marker`: everything below 75 becomes marker `2`, everything at
> or above becomes marker `1`. It still works here because coins are the darker class
> (see the review notes), but it is not doing what the two separate thresholds suggest —
> worth keeping in mind before reusing this on a dataset with different contrast.

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology
from skimage.color import label2rgb
from skimage.filters import sobel
from skimage.measure import regionprops
from skimage.segmentation import watershed

In [ ]:
def detect_regions(image, threshold=75, dilation_disk=1, erosion_disk=18, min_area=25):
    markers = np.zeros_like(image)
    markers[image >= threshold] = 1
    markers[image < threshold] = 2

    elevation_map = sobel(image)
    segmentation = watershed(elevation_map, markers)

    dilated = morphology.dilation(segmentation, morphology.disk(dilation_disk))
    filled = ndi.binary_fill_holes(dilated - 1) # keep marker 2 (coins) as foreground
    eroded = morphology.erosion(filled, morphology.disk(erosion_disk))

    labeled, _ = ndi.label(eroded)
    props = regionprops(labeled)
    count = sum(1 for r in props if r.area >= min_area)
    return count, labeled

## Visual check

In [ ]:
for name in sorted(images)[:3]:
    image = images[name]
    gt = common.real_count(df, name)

    elevation_map = sobel(image)
    common.P(image, 'original', size=6, cmap='gray')
    common.P(elevation_map, 'elevation map', size=6, cmap='gray')

    count, labeled = detect_regions(image)
    result = label2rgb(labeled, image=image)
    common.P(result, f'Real: {gt} | Found: {count}', size=6)
    common.S()

# Evaluation
Run the detector over the whole prepared dataset, score it against `real_count`, and
write a summary to `results/regions_results.yaml` (or `regions_results-full.yaml` when
`dataset_size == 'full'`).

In [ ]:
results, summary = common.evaluate_method(
    images, df,
    lambda image: detect_regions(image)[0],
    method_name='Watershed regions',
    parameters={'low_marker': 50, 'high_marker': 75, 'dilation_disk': 1, 'erosion_disk': 18, 'min_area': 25},
    results_path=base_folder / 'results' / f'regions_results{"" if dataset_size == "small" else "-full"}.yaml',
)

for key, value in summary.items():
    print(f'{key}: {value}')

In [ ]:
results[['abs_error', 'time_seconds', 'peak_memory_kb']].describe()